In [2]:
target_path = '/home/gina101/new_data/Ljubljana/Environment/Session2'

In [3]:
import fnmatch
import os
import pandas as pd
import shutil
import itertools

In [6]:
# source_path = r'/content/drive/MyDrive/TT8/Sociodrama 6/MAX - superepisodes'
source_path = '/home/gina101/Session2'
ids_path = os.path.join(source_path, 'Scene_IDs_XLSX')
scenes_path = os.path.join(source_path, 'Scenes_XLSX')
tasks_path = os.path.join(source_path, 'Tasks_XLSX')


# Extract Scene Data
1. Find ID file for each scene
2. Find line that corresponds to each timestamp
3. Find all task files that this scene belongs to
4. 'Splice' and only keep these files
5. Join them together in one dataframe
6. Write them to a file

In [7]:
def findSensorNoInTaskFiles(name):
  # denominators = ["HR", "SC", "Temp"]
  # for denom in denominators:
  #   if denom in name:
  #     parts = name.split(denom)
  #     if len(parts) > 1:
  #         # return parts[1].split('-')[0]
  #         return parts[1].split('.xlsx')[0]
  return name.split(".xlsx")[0].split("Sensor")[1]
  # return None

# def sceneNo(name):
#   denominators = ["HR", "SC", "Temp"]
#   for denom in denominators:
#     if denom in name:
#       parts = name.split(denom)
#       if len(parts) > 1:
#         return parts[0].split("Scene")[1]

def sceneNo(name):
  return name.split("_")[0].split("Scene")[1]

In [8]:
def findTaskNo(name):
  return name.split("Task")[1].split('.xlsx')[0]

In [13]:
# Find Scene ID files
id_files = os.listdir(ids_path)
scene_files = os.listdir(scenes_path)
task_files = os.listdir(tasks_path)

# Print these files
print("All idScene files found: ", id_files)
print("All task files found: ", task_files)

processed_scenes = []

def get_sensor_type(filename):
    if "HR" in filename: return "HR"
    if "SC" in filename: return "SC"
    if "Temp" in filename: return "Temp"
    return None

for id_scene_file in id_files:
  all_filtered_task_files = []
  # Find Scene No and Task No from the id_scene_file
  scene_num = sceneNo(id_scene_file)
  if scene_num in processed_scenes:
    continue
  processed_scenes.append(scene_num)

  task_num = findTaskNo(id_scene_file)
  print(f"Processing idScene file: {id_scene_file} (Scene no: {scene_num}, Task no: {task_num})")

  task_files_for_this_scene = fnmatch.filter(task_files, "Task{}*".format(task_num))
  # print(f"Task files for this scene: {task_files_for_this_scene}")
  assert(len(task_files_for_this_scene) == 12)

  # Open id_scene_file and read cells A1 and A2
  id_scene_file_path = os.path.join(ids_path, id_scene_file)
  df_id_scene = pd.read_excel(id_scene_file_path, header=None)
  # print(df_id_scene)
  starting_timestamp = df_id_scene.iloc[0, 0].astype(int)
  ending_timestamp = df_id_scene.iloc[1, 0].astype(int)

  # # Now, time to find the appropriate lines to 'splice' the files
  # # Read through each one of the task files and find the lines that have the values of cell_a1 and cell_a2

  for task_file in task_files_for_this_scene:
      task_file_path = os.path.join(tasks_path, task_file)
      df_task = pd.read_excel(task_file_path)

      # Convert the first column of df_task to integer type for comparison
      start_index_series = df_task[df_task.iloc[:, 0].astype(int) == starting_timestamp].index
      if start_index_series.empty:
        print("Start index empty")
      end_index_series = df_task[df_task.iloc[:, 0].astype(int) == ending_timestamp].index
      if end_index_series.empty:
        print("End index empty")

      if not start_index_series.empty and not end_index_series.empty:
          start_row = start_index_series[0]
          end_row = end_index_series[0]
          if start_row <= end_row:
            # Create new dataframe and copy the rows
            spliced_df = df_task.iloc[start_row : end_row + 1]

            # Extract sensor type and number for column naming
            sensor_type = get_sensor_type(task_file)
            sensor_num = findSensorNoInTaskFiles(task_file)
            print(f"Task file {task_file} has sensor type {sensor_type} and belongs to participant {sensor_num}")

            if sensor_type and sensor_num:
              # Assuming the first column is 'Timestamp' and the second is the value
              spliced_df.columns = ['Timestamp', 'Value']

          else:
            print(f"Warning: Start index ({start_row}) is greater than end index ({end_row}) for {task_file}. Skipping splicing.")

          output_filename = f"Scene{scene_num}_{sensor_type}Sensor{sensor_num}_Task{task_num}.xlsx"
          output_path = os.path.join(target_path, output_filename)
          spliced_df.to_excel(output_path, index=False)
          print(f"Saved spliced data for {task_file} to {output_path}")
      else:
        print(f"No data merged for Scene {scene_num}, Task {task_num}")

All idScene files found:  ['idScene6_TempSensor3_Task6.xlsx', 'idScene2_SCSensor4_Task2.xlsx', 'idScene4_SCSensor4_Task5.xlsx', 'idScene2_HRSensor2_Task2.xlsx', 'idScene1_SCSensor4_Task1.xlsx', 'idScene1_SCSensor1_Task1.xlsx', 'idScene3_SCSensor3_Task4.xlsx', 'idScene6_SCSensor4_Task6.xlsx', 'idScene1_TempSensor2_Task1.xlsx', 'idScene3_HRSensor2_Task4.xlsx', 'idScene1_SCSensor3_Task1.xlsx', 'idScene5_TempSensor4_Task6.xlsx', 'idScene2_TempSensor1_Task2.xlsx', 'idScene2_TempSensor4_Task2.xlsx', 'idScene2_SCSensor3_Task2.xlsx', 'idScene1_TempSensor3_Task1.xlsx', 'idScene5_TempSensor2_Task6.xlsx', 'idScene3_TempSensor4_Task4.xlsx', 'idScene2_SCSensor2_Task2.xlsx', 'idScene4_HRSensor2_Task5.xlsx', 'idScene4_SCSensor3_Task5.xlsx', 'idScene3_TempSensor3_Task4.xlsx', 'idScene6_SCSensor3_Task6.xlsx', 'idScene1_SCSensor2_Task1.xlsx', 'idScene1_TempSensor1_Task1.xlsx', 'idScene1_TempSensor4_Task1.xlsx', 'idScene5_SCSensor2_Task6.xlsx']
All task files found:  ['Task6HRSensor4.xlsx', 'Task1SCSenso

In [9]:
def get_sensor_type(filename):
    if "HR" in filename: return "HR"
    if "SC" in filename: return "SC"
    if "Temp" in filename: return "Temp"
    return None

In [13]:
new_scene_folder = os.path.join(source_path, "Scenes_Complete_Gina")
scene_files = os.listdir(new_scene_folder)
sonifications_path = os.path.join(target_path, "Sonifications")
if not os.path.exists(sonifications_path):
  os.makedirs(sonifications_path)

processed_scenes = []
for f in scene_files:
  scene_num = sceneNo(f)
  if scene_num not in processed_scenes:
    processed_scenes.append(scene_num)
  else:
    continue
  # If it doesn't exist already, create a folder for this scene
  scene_folder = os.path.join(sonifications_path, "Scene{}".format(scene_num))
  if not os.path.exists(scene_folder):
    os.makedirs(scene_folder)
  scene_files_filtered = fnmatch.filter(scene_files, "Scene{}*".format(scene_num))
  assert(len(scene_files_filtered) == 12)
  # Find all different sensor numbers in Scene_files_filtered. Sensors are like so in the file name: _SensorX_, where X = sensor_num
  sensor_nums = []
  for f in scene_files_filtered:
    sensor_num = f.split("_")[1].split("Sensor")[1]
    if sensor_num not in sensor_nums:
      sensor_nums.append(sensor_num)
  for sensor in sensor_nums:
    sensor_files_per_scene = fnmatch.filter(scene_files_filtered, "*Sensor{}*".format(sensor))
    assert(len(sensor_files_per_scene) == 3)
    # Create a new dataframe that concatenates these three files into a new one, removing the timestamp column and heading each new column with the appropriate bio code: "HR", "SC", "Temp"

    # Ensure the files are concatenated in HR -> SC -> Temp order
    order = {"HR": 0, "SC": 1, "Temp": 2}
    sensor_files_per_scene_sorted = sorted(sensor_files_per_scene, key=lambda x: order.get(get_sensor_type(x), 99))

    dfs = []
    for f in sensor_files_per_scene_sorted:
      df = pd.read_excel(os.path.join(new_scene_folder, f))
      values = df.iloc[:, 1]
      df_clean = pd.DataFrame({get_sensor_type(f): values})
      dfs.append(df_clean)

    print(f"Preview for Scene {scene_num} Sensor {sensor}")
    # Concatenate into a final dataframe horizontally
    final_df = pd.concat(dfs, axis=1)
    # Print first few rows to ensure correctness
    print(final_df.head())

    # Save to output file
    out_name = f"Scene{scene_num}AllSensor{sensor}.csv"
    final_path = os.path.join(scene_folder, out_name)
    final_df.to_csv(final_path, index=False)
    print(f"Saved to {final_path}")


Preview for Scene 4 Sensor 4
           HR        SC      Temp
0  202.105263  0.498923  32.65648
1  202.105263  0.498834  32.65648
2  202.105263  0.498923  32.65648
3  202.105263  0.499012  32.65648
4  202.105263  0.499278  32.65648
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Sonifications/Scene4/Scene4AllSensor4.csv
Preview for Scene 4 Sensor 1
          HR        SC       Temp
0  93.658537  1.477656  30.780236
1  93.658537  1.477656  30.780236
2  93.658537  1.477363  30.780236
3  93.658537  1.477363  30.780236
4  93.658537  1.477363  30.780236
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Sonifications/Scene4/Scene4AllSensor1.csv
Preview for Scene 4 Sensor 3
          HR        SC       Temp
0  93.249369  3.233553  31.109993
1  93.160857  3.234139  31.109993
2  93.072372  3.234432  31.109993
3  92.983917  3.235018  31.109993
4  92.895492  3.235018  31.109993
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Sonifications/Scene4/Scene4A

In [12]:
visualizations_path = os.path.join(target_path, 'Visualizations')
if not os.path.exists(visualizations_path):
  os.makedirs(visualizations_path)

task_files = os.listdir(tasks_path)

processed_tasks = []
for f in task_files:
  sensor_type = get_sensor_type(f)
  print('Sensor type: ', sensor_type)
  task_num = f.split(sensor_type)[0].split('Task')[1]
  print('Task number: ', task_num)
  if task_num not in processed_tasks:
    processed_tasks.append(task_num)
  else:
    continue

  # If it doesn't exist already, create a folder for this scene
  task_folder = os.path.join(visualizations_path, "Task{}".format(task_num))
  if not os.path.exists(task_folder):
    os.makedirs(task_folder)
  task_files_filtered = fnmatch.filter(task_files, "Task{}*".format(task_num))
  print(task_files_filtered)
  assert(len(task_files_filtered) == 12)
  # Find all different sensor numbers in task_files_filtered. Sensors are like so in the file name: *SensorX.xlsx, where X = sensor_num
  sensor_nums = []
  for f in task_files_filtered:
    print(f)
    sensor_num = f.split(".xlsx")[0].split("Sensor")[1]
    if sensor_num not in sensor_nums:
      sensor_nums.append(sensor_num)
  for sensor in sensor_nums:
    sensor_files_per_task = fnmatch.filter(task_files_filtered, "*Sensor{}*".format(sensor))
    assert(len(sensor_files_per_task) == 3)

    # Ensure the files are concatenated in HR -> SC -> Temp order
    order = {"HR": 0, "SC": 1, "Temp": 2}
    sensor_files_per_task_sorted = sorted(sensor_files_per_task, key=lambda x: order.get(get_sensor_type(x), 99))

    dfs = []
    for f in sensor_files_per_task_sorted:
      df = pd.read_excel(os.path.join(tasks_path, f))
      values = df.iloc[:, 1]
      df_clean = pd.DataFrame({get_sensor_type(f): values})
      dfs.append(df_clean)

    print(f"Preview for Task {task_num} Sensor {sensor}")
    # Concatenate into a final dataframe horizontally
    final_df = pd.concat(dfs, axis=1)

    # Enforce column order HR -> SC -> Temp if present
    desired_cols = [c for c in ["HR", "SC", "Temp"] if c in final_df.columns]
    final_df = final_df[desired_cols]
    # Print first few rows to ensure correctness
    print(final_df.head())

    # Save to output file
    out_name = f"Task{task_num}AllSensor{sensor}.csv"
    final_path = os.path.join(task_folder, out_name)
    final_df.to_csv(final_path, index=False)
    print(f"Saved to {final_path}")


Sensor type:  HR
Task number:  6
['Task6HRSensor4.xlsx', 'Task6TempSensor2.xlsx', 'Task6SCSensor4.xlsx', 'Task6TempSensor4.xlsx', 'Task6TempSensor1.xlsx', 'Task6SCSensor3.xlsx', 'Task6HRSensor3.xlsx', 'Task6TempSensor3.xlsx', 'Task6SCSensor2.xlsx', 'Task6HRSensor2.xlsx', 'Task6HRSensor1.xlsx', 'Task6SCSensor1.xlsx']
Task6HRSensor4.xlsx
Task6TempSensor2.xlsx
Task6SCSensor4.xlsx
Task6TempSensor4.xlsx
Task6TempSensor1.xlsx
Task6SCSensor3.xlsx
Task6HRSensor3.xlsx
Task6TempSensor3.xlsx
Task6SCSensor2.xlsx
Task6HRSensor2.xlsx
Task6HRSensor1.xlsx
Task6SCSensor1.xlsx
Preview for Task 6 Sensor 4
          HR        SC       Temp
0  93.658537  0.603352  34.186254
1  93.658537  0.603352  34.186254
2  93.658537  0.603352  34.186254
3  93.658537  0.603175  34.186254
4  93.658537  0.602997  34.186254
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task6/Task6AllSensor4.csv
Preview for Task 6 Sensor 2
          HR        SC       Temp
0  70.437510  2.009231  31.032687
1 

Final thing I forgot; we should trim the files so that each column has the same number of lines!

In [14]:
for folder in os.listdir(target_path):
  folder_path = os.path.join(target_path, folder)
  print('Folder: ', folder_path)
  for subfolder in os.listdir(folder_path):
    subfolder_path = os.path.join(folder_path, subfolder)
    print(f'Subfolder: {subfolder_path}')
    for f in os.listdir(subfolder_path):
      # Here we are either accessing scene or task files
      file_path = os.path.join(subfolder_path, f)
      print(f'Final path: {file_path}')
      # Read each csv
      df = pd.read_csv(file_path)
      # Find the length of each column
      lengths = [len(df[col]) for col in df.columns]
      # Find the minimum length
      min_length = min(lengths)
      # Trim the other two columns to have the same no. of lines
      for col in df.columns:
        if len(df[col]) != min_length:
          df = df.drop(df.tail(len(df[col]) - min_length).index)
      # print end to ensure it's correct
      print(df.tail())
      # Write output to same file
      df.to_csv(file_path, index=False)


Folder:  /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations
Subfolder: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2
Final path: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2/Task2AllSensor3.csv
               HR        SC       Temp
21089  112.941176  3.145934  31.016464
21090  112.941176  3.146227  31.016464
21091  112.941176  3.146813  31.016464
21092  112.941176  3.146813  31.016464
21093  112.941176  3.146813  31.016464
Final path: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2/Task2AllSensor4.csv
               HR        SC       Temp
21113  182.857143  2.461099  32.582459
21114  182.857143  2.460806  32.582459
21115  182.857143  2.460806  32.582459
21116  182.857143  2.460806  32.582459
21117  182.857143  2.460806  32.582459
Final path: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2/Task2AllSensor2.csv
              HR        SC       Temp
21092  73

## From this point forward, this code is usable for Eleusina

In [ ]:
def findSection(name):
  return name.split("-")[1].split('.xlsx')[0]

In [ ]:
current_folder = os.path.join(source_path, 'Tasks')
file_list = [n for n in os.listdir(current_folder) if not fnmatch.fnmatch(n, "time*")]

In [ ]:
sections = {}
for f in file_list:
  section = findSection(f)
  task = findTaskNo(f)
  if section not in sections:
    sections[section] = {task: [f]}
  else:
    if task in sections[section]:
      sections[section][task].append(f)
    else:
      sections[section][task] = [f]

In [ ]:
sections

In [ ]:
for section in sections.keys():
  for task in sections[section].keys():
    sensors = []
    for f in sections[section][task]:
      sensor = findSensorNo(f)
      if sensor not in sensors:
        sensors.append(sensor)
    for sensor in sensors:
      sensor_files = fnmatch.filter(sections[section][task], '*Sensor{}*'.format(sensor))
      [hr_file] = fnmatch.filter(sensor_files, "*HR*")
      [sc_file] = fnmatch.filter(sensor_files, "*SC*")
      [temp_file] = fnmatch.filter(sensor_files, "*Temp*")
      hr_pd = pd.read_excel(os.path.join(current_folder, hr_file))
      sc_pd = pd.read_excel(os.path.join(current_folder, sc_file))
      temp_pd = pd.read_excel(os.path.join(current_folder, temp_file))
      res_pd = pd.concat([hr_pd, sc_pd, temp_pd], axis=1)
      new_file_name = 'Task{}AllSensor{}-{}.csv'.format(task, sensor, section)
      new_path = os.path.join(target_path, new_file_name)
      res_pd.to_csv(new_path, index=False, header=['HR', 'SC', 'Temp'])

# Trimming the ends of tasks that have a scene at the end that continues to the next task

In [ ]:
session_task_files_path = r'/content/drive/MyDrive/TT8/new_data/Thematic3/Session2/Visualizations'

Detection mechanism

In [ ]:
# Split scenes:
scenes = [21, 105]

In [ ]:
# First task in each scene
first_tasks = {
    21: ['Section 2', 2],
    105: ['Section 3', 3]
}

Find the ending line

In [ ]:
ending_rows = {
    21: 134752,
    105: 29413
}

Trim the files with different line counts accordingly.

1. Find files of this task in visualization folder
2. Find files that have lines greater than the one in ending_rows.
3. Remove lines
4. Save
5. Redownload



In [ ]:
for scene in first_tasks.keys():
  section_folder = os.path.join(session_task_files_path, first_tasks[scene][0])
  task_folder = os.path.join(section_folder, 'Task{}'.format(first_tasks[scene][1]))
  files = os.listdir(task_folder)
  for f in files:
    file_path = os.path.join(task_folder, f)
    df = pd.read_csv(file_path, header=0)
    diff = df.shape[0] - ending_rows[scene]
    if df.shape[0] > ending_rows[scene]:
      print('LARGE')
      print(f)
      df.drop(df.tail(diff).index, inplace = True)
      df.to_csv(file_path, index=False)
      print(df.shape[0])
    else:
      print('BASELINE')
      print(df.shape[0])

BASELINE
134752
BASELINE
134752
BASELINE
134752
BASELINE
134752
BASELINE
134752
BASELINE
29413
BASELINE
29413
BASELINE
29413
BASELINE
29413
BASELINE
29394
